# EDL-ECOC for Skin Lesion Classification with Uncertainty-Based OOD Detection

Evidential Deep Learning (EDL) combined with Error-Correcting Output Codes (ECOC),
evaluated on ISIC 2019.

**Setup**
- Backbone: EfficientNetV2-M (timm, ImageNet-21k pretrained, fine-tuned on 1k)
- Codebook: 28 pairwise (one-vs-one) heads + 8 one-vs-rest heads = 36 heads
- Train/val split: `StratifiedGroupKFold` grouped by `lesion_id`, so no lesion appears in both splits
- Test: the official ISIC 2019 test set. Its `UNK` class is treated as out-of-distribution

**Evaluation**
- Accuracy, macro-F1 and balanced accuracy on the 8 known classes
- UNK-detection AUROC from per-sample uncertainty
- Uncertainty-based pruning analysis

**Data.** Download ISIC 2019 (training input, test input, ground truth, metadata) and
set data_dir in the config cell below.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import random
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler

from typing import List, Optional, Sequence, Tuple, Dict
from itertools import combinations


from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import timm
from tqdm import tqdm

# Preprocessing

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()


SEED = 43

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

data_dir            = "/path/to/ISIC_DATA"
train_dir            = "/path/to/ISIC_DATA/ISIC_2019_Training_Input"
test_dir             = "/path/to/ISIC_DATA/ISIC_2019_Test_Input"
train_truth_file     = "ISIC_2019_Training_GroundTruth.csv"
test_truth_file      = "ISIC_2019_Test_GroundTruth.csv"
train_metadata_file  = "ISIC_2019_Training_Metadata.csv"

# -------------------------------------------------------------- classes
CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES = len(CLASS_NAMES)
UNK_LABEL = NUM_CLASSES

# ----------------------------------------------------------------- data
IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4

# ---------------------------------------------------------------- model
ARCH = "tf_efficientnetv2_m.in21k_ft_in1k"
DROPOUT = 0.4
DROP_PATH_RATE = 0.2
# k=1 -> pairwise (1-vs-1) heads, k=7 -> one-vs-rest heads  => 36 heads total
CODEBOOK_K_VALUES = (1, 7)

# ------------------------------------------------------------- training
NUM_EPOCHS = 35
LR = 1e-4
WEIGHT_DECAY = 0.01

# ----------------------------------------------------------------- loss
KL_WEIGHT = 1.0
KL_ANNEAL_EPOCHS = 10
LAMBDA_OOD = 0.4
OOD_ANNEAL_EPOCHS = 10


In [ ]:
train_gt = pd.read_csv(os.path.join(data_dir, train_truth_file))
test_gt  = pd.read_csv(os.path.join(data_dir, test_truth_file))

# Create single_label for training (exclude image & UNK)
train_labels = train_gt.iloc[:, 1:9]
train_gt['single_label'] = train_labels.values.argmax(axis=1)

# For test (include UNK)
test_labels = test_gt.iloc[:, 1:10]
test_gt['single_label'] = test_labels.values.argmax(axis=1)


meta = pd.read_csv(os.path.join(data_dir, train_metadata_file))

# Merge lesion_id from metadata into train_gt (left join, keyed on image name)
train_gt = train_gt.merge(meta[['image', 'lesion_id']], on='image', how='left')
train_gt['group_id'] = train_gt['lesion_id'].fillna(train_gt['image'])

# Stratified GROUP Train/Val Split

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(
    sgkf.split(train_gt, train_gt['single_label'], groups=train_gt['group_id'])
)

train_df = train_gt.iloc[train_idx].reset_index(drop=True).drop(columns=['group_id', 'lesion_id'])
val_df   = train_gt.iloc[val_idx].reset_index(drop=True).drop(columns=['group_id', 'lesion_id'])

train_groups = set(train_gt.loc[train_idx, 'group_id'])
val_groups   = set(train_gt.loc[val_idx, 'group_id'])
overlap = train_groups & val_groups
print(f"Overlapping groups between train/val: {len(overlap)}")

In [ ]:
class CenterSquareCrop:
    def __call__(self, img):
        w, h = img.size
        m = min(w, h)
        left, top = (w-m)//2, (h-m)//2
        return img.crop((left, top, left+m, top+m))


IMAGENET_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STD = [0.5, 0.5, 0.5]

IMAGENET_MEAN_FILL = tuple(
    round(x * 255) for x in IMAGENET_MEAN
)

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    CenterSquareCrop(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees = 30, translate=(0.01,0.01), scale=(1.0,1.05), fill=IMAGENET_MEAN_FILL),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    CenterSquareCrop(),
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])


class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        name = self.df.iloc[idx]['image'] + '.jpg'
        img = Image.open(os.path.join(self.img_dir, name))
        label = int(self.df.iloc[idx]['single_label'])
        if self.transform:
            img = self.transform(img)
        return img, label


train_ds = ISICDataset(train_df, train_dir, train_transform)
val_ds   = ISICDataset(val_df,   train_dir, val_transform)
test_ds  = ISICDataset(test_gt,  test_dir,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, worker_init_fn=seed_worker, generator=g,)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,num_workers=4, pin_memory=True)

## Training Routine

### Codebook

In [ ]:
Spec = Tuple[str, List[str], List[str]]  # (head_name, positive_classes, negative_classes)

def make_one_vs_k_specs(class_names: Sequence[str], k: int) -> List[Spec]:
   
    C = len(class_names)
    if not 1 <= k <= C - 1:
        raise ValueError(f"k must be in [1, {C - 1}], got {k}")

    class_order = {c: i for i, c in enumerate(class_names)}
    specs: List[Spec] = []

    for pos in class_names:
        remaining = [c for c in class_names if c != pos]
        for neg_subset in combinations(remaining, k):
            neg_sorted = sorted(neg_subset, key=class_order.__getitem__)
            name = f"1v{k}_{pos}_vs_{'-'.join(neg_sorted)}"
            specs.append((name, [pos], list(neg_sorted)))

    return specs


def build_codebook(
    class_names: Sequence[str],
    k_values: Sequence[int] = (7, 6, 5),
) -> Tuple[torch.Tensor, List[str]]:

    C = len(class_names)
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    specs: List[Spec] = []
    for k in k_values:
        specs += make_one_vs_k_specs(class_names, k)

    columns: List[torch.Tensor] = []
    head_names: List[str] = []
    seen: set = set()

    for head_name, pos_classes, neg_classes in specs:
        col = torch.zeros(C, dtype=torch.float32)
        for c in pos_classes:
            col[class_to_idx[c]] = 1.0
        for c in neg_classes:
            col[class_to_idx[c]] = -1.0

        key = tuple(col.tolist())
        flipped_key = tuple((-col).tolist())
        if key in seen or flipped_key in seen:
            continue

        seen.add(key)
        columns.append(col)
        head_names.append(head_name)

    codebook = torch.stack(columns, dim=1)  # [C, H]
    return codebook, head_names


### Model

In [ ]:
class EfficientNetEDLECOCModel(nn.Module):
    def __init__(
        self,
        codebook: torch.Tensor,           # [C, L] ternary +1/-1/0
        column_names: Optional[Sequence[str]] = None,
        arch: str = 'tf_efficientnetv2_m.in21k_ft_in1k',
        pretrained: bool = True,
        dropout: float = 0.2,
        drop_path_rate: float = 0.0,

    ):
        super().__init__()

        codebook = torch.as_tensor(codebook, dtype=torch.float32)
        if codebook.ndim != 2:
            raise ValueError(f"codebook must be 2D [num_classes, code_length], got {tuple(codebook.shape)}")
        unique_vals = set(codebook.unique().tolist())
        if not unique_vals.issubset({-1.0, 0.0, 1.0}):
            raise ValueError(f"codebook must contain only -1, 0, +1. Found: {sorted(unique_vals)}")

        self.num_classes = int(codebook.shape[0])
        self.code_length = int(codebook.shape[1])


        if column_names is None:
            column_names = [f"ECOC_{i:03d}" for i in range(self.code_length)]
        if len(column_names) != self.code_length:
            raise ValueError("column_names length must match code_length.")
        self.column_names: List[str] = list(column_names)

        self.register_buffer("codebook_pm1", codebook)
        self.register_buffer("class_active_counts", (codebook != 0).float().sum(dim=1).clamp_min(1.0))


        self.backbone = timm.create_model(
            arch,
            pretrained=pretrained,
            features_only=True,
            out_indices=(-1,),
            drop_path_rate=drop_path_rate,
        )
        self.feat_dim = self.backbone.feature_info.info[-1]['num_chs']
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        print(f"Backbone feature dim: {self.feat_dim}")
        self.bit_heads = nn.ModuleList([
            self._make_bit_head(dropout) for _ in range(self.code_length)
        ])

    def _make_bit_head(self, dropout: float) -> nn.Module:
        return nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feat_dim, 2))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)[-1]
        return self.pool(feat).float()

    def forward_bit_logits(self, x: torch.Tensor) -> torch.Tensor:
        features = self.extract_features(x)
        logits = [head(features).unsqueeze(1) for head in self.bit_heads]
        return torch.cat(logits, dim=1)  # [B, L, 2]

    def logits_to_evidence(self, logits: torch.Tensor) -> torch.Tensor:
        return F.softplus(logits)

    def logits_to_alpha(self, bit_logits: torch.Tensor) -> torch.Tensor:
        return self.logits_to_evidence(bit_logits) + 1.0

    def get_output(self, x: torch.Tensor) -> torch.Tensor:
        # returns per-head alpha [B, L, 2].
        return self.logits_to_alpha(self.forward_bit_logits(x))

    def predict_bit_probs_from_alpha(self, alpha: torch.Tensor) -> torch.Tensor:
        alpha_sum = alpha.sum(dim=-1).clamp_min(1e-8)
        return alpha[..., 1] / alpha_sum

    def predict_uncertainty_from_alpha(self, alpha: torch.Tensor) -> torch.Tensor:
        # Per-head vacuity: 2/S, bounded [0,1] since each head is a 2-way Dirichlet.
        alpha_sum = alpha.sum(dim=-1).clamp_min(1e-8)
        return (2.0 / alpha_sum).clamp(0.0, 1.0)

    def decode_bit_probs(self, p_pos: torch.Tensor) -> torch.Tensor:
        if p_pos.shape[1] != self.code_length:
            raise ValueError(f"p_pos must have shape [B, {self.code_length}], got {tuple(p_pos.shape)}")
        soft_bits = 2.0 * p_pos - 1.0
        class_scores = soft_bits @ self.codebook_pm1.T
        class_scores = class_scores / self.class_active_counts.view(1, -1)
        return class_scores

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        alpha = self.get_output(x)
        p_pos = self.predict_bit_probs_from_alpha(alpha)
        return self.decode_bit_probs(p_pos)



In [ ]:
class EDLECOCLoss(nn.Module):

    def __init__(
        self,
        codebook_pm1: torch.Tensor,
        kl_weight: float = 1.0,
        kl_anneal_epochs: int = 10,
        lambda_ood: float = 0.5,
        ood_anneal_epochs: int = 10,
        head_reduction: str = "mean",
        eps: float = 1e-8,

    ):
        super().__init__()

        codebook_pm1 = torch.as_tensor(codebook_pm1, dtype=torch.float32)
        if codebook_pm1.ndim != 2:
            raise ValueError(f"codebook_pm1 must be [C, L], got {tuple(codebook_pm1.shape)}")
        if head_reduction not in {"sum", "mean"}:
            raise ValueError("head_reduction must be 'sum' or 'mean'.")


        self.register_buffer("codebook_pm1", codebook_pm1)
        self.num_classes = codebook_pm1.shape[0]
        self.code_length = codebook_pm1.shape[1]

        self.kl_weight = kl_weight
        self.kl_anneal_epochs = kl_anneal_epochs
        self.lambda_ood = lambda_ood
        self.ood_anneal_epochs = ood_anneal_epochs
        self.head_reduction = head_reduction
        self.eps = eps

    
        self.register_buffer("uniform_prior", torch.ones(self.code_length, 2, dtype=torch.float32))

        n_active = (codebook_pm1 != 0).sum(dim=0)
        self.register_buffer("has_inactive_head", n_active < self.num_classes)

        if not bool(self.has_inactive_head.any()):
            warnings.warn(
                "No head in this codebook has an inactive class -- the local-OOD "
                "term will be zero everywhere and lambda_ood has no effect."
            )


    @classmethod
    def from_model(cls, model: "EfficientNetEDLECOCModel", **kwargs) -> "EDLECOCLoss":
        return cls(codebook_pm1=model.codebook_pm1.detach().cpu(), **kwargs)


    # Target encoding

    def encode_targets(
        self, y: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
          target_01  [B, L] in {0,1}: which side of each head is "positive"
                     for the sample's true class (only meaningful where active).
          active_mask [B, L] bool: head h is active (not don't-care) for
                     sample i's true class.
          valid      [B] bool: whether y[i] is a real, in-range class index.
          y_safe     [B] long: y clamped into [0, C-1], safe for indexing.
        """
        y = y.long()
        valid = (y >= 0) & (y < self.num_classes)
        y_safe = y.clamp(0, self.num_classes - 1)
        target_pm1 = self.codebook_pm1[y_safe]
        active_mask = (target_pm1 != 0) & valid.view(-1, 1)
        target_01 = (target_pm1 > 0).long()
        return target_01, active_mask, valid, y_safe


    # Core evidential terms

    def edl_data_fit(self, target_onehot: torch.Tensor, alpha: torch.Tensor) -> torch.Tensor:
        alpha = alpha.clamp_min(self.eps)
        S = alpha.sum(dim=-1, keepdim=True).clamp_min(self.eps)
        return (target_onehot * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=-1)

    def dirichlet_kl_masked(
        self, alpha: torch.Tensor, target_onehot: torch.Tensor, prior: torch.Tensor
    ) -> torch.Tensor:
        """Eq. (7): KL(Dir(alpha_tilde) || prior); alpha_tilde clamps the TRUE
        side to its prior value so only wrong-side evidence is penalized."""
        alpha_tilde = (target_onehot + (1.0 - target_onehot) * alpha).clamp_min(self.eps)
        prior = prior.clamp_min(self.eps)
        sum_a, sum_p = alpha_tilde.sum(-1, keepdim=True), prior.sum(-1, keepdim=True)
        kl = (
            torch.lgamma(sum_a) - torch.lgamma(alpha_tilde).sum(-1, keepdim=True)
            - torch.lgamma(sum_p) + torch.lgamma(prior).sum(-1, keepdim=True)
            + ((alpha_tilde - prior) * (torch.digamma(alpha_tilde) - torch.digamma(sum_a))).sum(-1, keepdim=True)
        )
        return kl.squeeze(-1)

    def dirichlet_kl_unmasked(self, alpha: torch.Tensor, prior: torch.Tensor) -> torch.Tensor:
        """Eq. (11): KL(Dir(alpha) || prior), unmasked -- used for local-OOD."""
        alpha = alpha.clamp_min(self.eps)
        prior = prior.clamp_min(self.eps)
        sum_a, sum_p = alpha.sum(-1, keepdim=True), prior.sum(-1, keepdim=True)
        kl = (
            torch.lgamma(sum_a) - torch.lgamma(alpha).sum(-1, keepdim=True)
            - torch.lgamma(sum_p) + torch.lgamma(prior).sum(-1, keepdim=True)
            + ((alpha - prior) * (torch.digamma(alpha) - torch.digamma(sum_a))).sum(-1, keepdim=True)
        )
        return kl.squeeze(-1)

    def _effective_weight(self, base: float, anneal_epochs: int, epoch: Optional[int]) -> float:
        if anneal_epochs <= 0 or epoch is None:
            return base
        return base * min(1.0, float(epoch) / float(anneal_epochs))

    def reduce_local_ood_by_class(
        self,
        kl_ood_per_head: torch.Tensor,   # [B, L]
        y: torch.Tensor,                 # [B]
        inactive_mask_f: torch.Tensor,   # [B, L], already includes has_inactive_head
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        device, dtype = kl_ood_per_head.device, kl_ood_per_head.dtype
        C = self.num_classes

        y_safe = y.long().clamp(0, C - 1)  # invalid rows are already zeroed via inactive_mask_f
        class_onehot = F.one_hot(y_safe, num_classes=C).to(device=device, dtype=dtype)  # [B, C]

        # [B, C, L]: sample i contributes to (class c, head h) iff class(i)==c
        # AND c is inactive for h.
        class_head_mask = class_onehot.unsqueeze(-1) * inactive_mask_f.to(dtype).unsqueeze(1)

        class_head_counts = class_head_mask.sum(dim=0)                              # [C, L] = |J_{h,c}|
        class_head_kl_sum = (class_head_mask * kl_ood_per_head.unsqueeze(1)).sum(dim=0)  # [C, L]
        class_head_mean_kl = class_head_kl_sum / class_head_counts.clamp_min(1.0)    # inner avg, Eq.(12)

        class_present = class_head_counts > 0                                        # [C, L]

        numerator   = (class_head_mean_kl * class_present.to(dtype)).sum(dim=0)    
        denominator = class_present.to(dtype).sum(dim=0)                                         # [L]
        local_ood_loss = torch.where(
            denominator > 0, numerator / denominator.clamp_min(self.eps), torch.zeros_like(numerator)
        )

        num_present_ood_classes = class_present.sum(dim=0)  # [L], diagnostic only
        return local_ood_loss, num_present_ood_classes

    # ------------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------------

    def forward(
        self, alpha: torch.Tensor, y: torch.Tensor, epoch: Optional[int] = None
    ) -> Tuple[torch.Tensor, Dict]:
        device, dtype = alpha.device, alpha.dtype
        target_01, active_mask, valid, y_safe = self.encode_targets(y)
        active_mask_f = active_mask.float().to(device)
        target_onehot = F.one_hot(target_01.to(device), num_classes=2).to(dtype)

        alpha = alpha.clamp_min(self.eps)

        # --- Primary term: Eqs. (5),(7),(9)-(10) -- fit + KL, masked to
        # heads active for the true class, averaged per-head then per-batch. ---
        fit_per_head = self.edl_data_fit(target_onehot, alpha)

        prior_batch = self.uniform_prior.to(device, dtype).unsqueeze(0).expand_as(alpha)
        eff_kl_weight = self._effective_weight(self.kl_weight, self.kl_anneal_epochs, epoch)
        kl_per_head = self.dirichlet_kl_masked(alpha, target_onehot, prior_batch)

        per_sample_head_loss = (fit_per_head + eff_kl_weight * kl_per_head) * active_mask_f
        per_head_active_count = active_mask_f.sum(dim=0).clamp_min(1.0)
        primary_active_head_mask = active_mask_f.sum(dim=0) > 0

        head_losses_all = per_sample_head_loss.sum(dim=0) / per_head_active_count
        head_fit_all = (fit_per_head * active_mask_f).sum(dim=0) / per_head_active_count
        head_kl_all = (kl_per_head * active_mask_f).sum(dim=0) / per_head_active_count

        inactive_mask_f = (~active_mask & valid.view(-1, 1).to(device)).float()
        inactive_mask_f = inactive_mask_f * self.has_inactive_head.to(device).float().unsqueeze(0)

        eff_lambda_ood = self._effective_weight(self.lambda_ood, self.ood_anneal_epochs, epoch)
        kl_ood_per_head = self.dirichlet_kl_unmasked(alpha, prior_batch)

        local_ood_loss_all, num_present_ood_classes = self.reduce_local_ood_by_class(
            kl_ood_per_head=kl_ood_per_head, y=y, inactive_mask_f=inactive_mask_f
        )

        ood_present_head_mask = num_present_ood_classes > 0
        include_head_mask = primary_active_head_mask | ood_present_head_mask

        head_losses_all = head_losses_all + eff_lambda_ood * local_ood_loss_all

        head_losses = head_losses_all[include_head_mask]
        head_fit = head_fit_all[include_head_mask]
        head_kl = head_kl_all[include_head_mask]
        head_ood = local_ood_loss_all[include_head_mask]
        head_ood_n_classes = num_present_ood_classes[include_head_mask]


        if head_losses.numel() == 0:
            total_loss = torch.zeros((), device=device, dtype=dtype)
            data_fit_reduced = kl_raw_reduced = ood_raw_reduced = torch.zeros((), device=device, dtype=dtype)
            mean_ood_classes = torch.zeros((), device=device, dtype=dtype)
        else:
            reduce_fn = torch.sum if self.head_reduction == "sum" else torch.mean
            total_loss = reduce_fn(head_losses)
            data_fit_reduced = reduce_fn(head_fit)
            kl_raw_reduced = reduce_fn(head_kl)
            ood_raw_reduced = reduce_fn(head_ood)
            mean_ood_classes = head_ood_n_classes.float().mean()

        logs = {
            "loss": total_loss.detach(),
            "kl_anneal_weight": eff_kl_weight,
            "ood_anneal_weight": eff_lambda_ood,
            "loss_data_fit": data_fit_reduced.detach(),
            "loss_kl_unweighted": kl_raw_reduced.detach(),
            "loss_kl_weighted": (eff_kl_weight * kl_raw_reduced).detach(),
            "loss_ood_unweighted": ood_raw_reduced.detach(),
            "loss_ood_weighted": (eff_lambda_ood * ood_raw_reduced).detach(),
            "ood_mean_present_classes": mean_ood_classes.detach(),  # diagnostic
        }
        return total_loss, logs

In [ ]:
def compute_ecoc_epoch_metrics(
    model: EfficientNetEDLECOCModel,
    criterion: EDLECOCLoss,
    alpha: torch.Tensor,
    y: torch.Tensor,
) -> Dict[str, float]:
    device = model.codebook_pm1.device
    alpha = alpha.to(device)
    y = y.to(device)

    L = model.code_length
    valid = (y >= 0) & (y < model.num_classes)

    p_pos = model.predict_bit_probs_from_alpha(alpha)
    pred_bit = (p_pos > 0.5).long()
    class_scores = model.decode_bit_probs(p_pos)
    preds = class_scores.argmax(dim=1)

    target_01, active_mask, _, _ = criterion.encode_targets(y)
    uncertainty = model.predict_uncertainty_from_alpha(alpha)   # [N, L]
    evidence = (alpha - 1.0).clamp_min(0.0).sum(dim=-1)         # [N, L]

    true_active_f = active_mask.float()
    pred_active_mask = (model.codebook_pm1[preds] != 0) & valid.view(-1, 1)
    pred_active_f = pred_active_mask.float()

    true_class_active_sum = max(true_active_f.sum().item(), 1.0)
    pred_class_active_sum = max(pred_active_f.sum().item(), 1.0)

    if valid.any():
        y_true = y[valid].cpu().numpy()
        y_pred = preds[valid].cpu().numpy()
        class_accuracy = accuracy_score(y_true, y_pred)
        class_macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        class_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    else:
        class_accuracy = class_macro_f1 = class_balanced_accuracy = 0.0

    pred_bit_np = pred_bit.cpu().numpy()
    target_01_np = target_01.cpu().numpy()
    active_mask_np = active_mask.cpu().numpy()
    uncertainty_np = uncertainty.cpu().numpy()

    head_bit_accuracy = np.full(L, np.nan)
    head_bit_f1 = np.full(L, np.nan)
    head_bit_balanced_accuracy = np.full(L, np.nan)
    head_uncertainty_calibration_auc = np.full(L, np.nan)

    for h in range(L):
        m = active_mask_np[:, h]
        if m.sum() < 2:
            continue
        y_true_h = target_01_np[m, h]
        y_pred_h = pred_bit_np[m, h]
        unc_h = uncertainty_np[m, h]

        head_bit_accuracy[h] = accuracy_score(y_true_h, y_pred_h)
        head_bit_f1[h] = f1_score(y_true_h, y_pred_h, zero_division=0)

        if len(np.unique(y_true_h)) < 2:
            continue
        head_bit_balanced_accuracy[h] = balanced_accuracy_score(y_true_h, y_pred_h)

        wrong_h = (y_pred_h != y_true_h).astype(float)
        if len(np.unique(wrong_h)) >= 2:
            try:
                head_uncertainty_calibration_auc[h] = roc_auc_score(wrong_h, unc_h)
            except ValueError:
                pass

    return {
        "class_accuracy": class_accuracy,
        "class_macro_f1": class_macro_f1,
        "class_balanced_accuracy": class_balanced_accuracy,
        "head_bit_accuracy_avg": float(np.nanmean(head_bit_accuracy)),
        "head_bit_f1_avg": float(np.nanmean(head_bit_f1)),
        "head_bit_balanced_accuracy_avg": float(np.nanmean(head_bit_balanced_accuracy)),
        "head_uncertainty_calibration_auc": float(np.nanmean(head_uncertainty_calibration_auc)),
        "true_class_mean_vacuity": (uncertainty * true_active_f).sum().item() / true_class_active_sum,
        "true_class_mean_evidence": (evidence * true_active_f).sum().item() / true_class_active_sum,
        "pred_class_mean_vacuity": (uncertainty * pred_active_f).sum().item() / pred_class_active_sum,
        "pred_class_mean_evidence": (evidence * pred_active_f).sum().item() / pred_class_active_sum,
        "global_mean_vacuity": uncertainty.mean().item(),
    }

In [ ]:
codebook, head_names = build_codebook(CLASS_NAMES, k_values=CODEBOOK_K_VALUES)

print(f"num_classes={codebook.shape[0]}, num_heads={codebook.shape[1]}")


model = EfficientNetEDLECOCModel(
    codebook=codebook,
    column_names=head_names,
    arch='tf_efficientnetv2_m.in21k_ft_in1k',
    pretrained=True,
    dropout=DROPOUT,
    drop_path_rate=DROP_PATH_RATE,
).to(device)

criterion = EDLECOCLoss.from_model(
    model,
    kl_weight=KL_WEIGHT,
    kl_anneal_epochs=KL_ANNEAL_EPOCHS,
    lambda_ood=LAMBDA_OOD,
    ood_anneal_epochs=OOD_ANNEAL_EPOCHS,
    head_reduction='mean',
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler()

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, epoch):
    model.train()
    total_loss = 0.0
    total_samples = 0
    all_alpha, all_y = [], []

    for inputs, targets in tqdm(loader, desc='Train'):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast():
            bit_logits = model.forward_bit_logits(inputs)

        alpha = model.logits_to_alpha(bit_logits.float())
        loss, _ = criterion(alpha, targets, epoch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * targets.size(0)
        total_samples += targets.size(0)

        with torch.no_grad():
            all_alpha.append(alpha.detach().cpu())
            all_y.append(targets.detach().cpu())

    avg_loss = total_loss / total_samples
    all_alpha = torch.cat(all_alpha, dim=0)
    all_y = torch.cat(all_y, dim=0)
    metrics = compute_ecoc_epoch_metrics(model, criterion, all_alpha, all_y)
    return avg_loss, metrics

@torch.no_grad()
def evaluate(model, loader, device, epoch):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_alpha, all_y = [], []

    for inputs, targets in tqdm(loader, desc='Eval'):
        inputs, targets = inputs.to(device), targets.to(device)
        alpha = model.get_output(inputs)
        loss, _ = criterion(alpha, targets, epoch)

        total_loss += loss.item() * targets.size(0)
        total_samples += targets.size(0)
        all_alpha.append(alpha.cpu())
        all_y.append(targets.cpu())

    avg_loss = total_loss / total_samples
    all_alpha = torch.cat(all_alpha, dim=0)
    all_y = torch.cat(all_y, dim=0)
    metrics = compute_ecoc_epoch_metrics(model, criterion, all_alpha, all_y)
    return avg_loss, metrics

In [ ]:
best_val_f1 = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device, epoch)
    val_loss, val_metrics = evaluate(model, val_loader, device, epoch)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_metrics['class_accuracy']*100:.2f}% | Train Macro F1: {train_metrics['class_macro_f1']:.4f}")
    print(f" Val  Loss: {val_loss:.4f} | Val  Acc: {val_metrics['class_accuracy']*100:.2f}% | Val  Macro F1: {val_metrics['class_macro_f1']:.4f}")
    print(f" Val  Head Bit Acc: {val_metrics['head_bit_accuracy_avg']:.4f} | Val Head Bit F1: {val_metrics['head_bit_f1_avg']:.4f} | Val Uncertainty-Calib AUC: {val_metrics['head_uncertainty_calibration_auc']:.4f}")

    val_f1 = val_metrics['class_macro_f1']
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'edl_ecoc_checkpoint.pth')
        print(f"Saved best model (F1={best_val_f1:.4f})")

print("Training complete.")

# Testing:

In [ ]:
model.load_state_dict(torch.load('edl_ecoc_checkpoint.pth', weights_only=True))
model.to(device)

In [ ]:
@torch.no_grad()
def get_predictions_and_uncertainty(model, alpha):

    device = model.codebook_pm1.device
    alpha = alpha.to(device)

    p_pos = model.predict_bit_probs_from_alpha(alpha)          # [N, L]
    class_scores = model.decode_bit_probs(p_pos)                # [N, C]
    preds = class_scores.argmax(dim=1)                          # [N]

    uncertainty_per_head = model.predict_uncertainty_from_alpha(alpha)  # [N, L]

    pred_active_mask = (model.codebook_pm1[preds] != 0).float()         # [N, L]
    pred_denom = pred_active_mask.sum(dim=1).clamp_min(1.0)             # [N], |H(pred)|
    uncertainty = (uncertainty_per_head * pred_active_mask).sum(dim=1) / pred_denom

    global_uncertainty = uncertainty_per_head.mean(dim=1)

    soft_bits = 2.0 * p_pos - 1.0                                        # [N, L]
    contrib_weight = soft_bits.abs() * pred_active_mask                  # [N, L]

    # Divide by active-head COUNT
    contribution_weighted_uncertainty = (
        (uncertainty_per_head * contrib_weight).sum(dim=1) / pred_denom
    )

    return preds, uncertainty, global_uncertainty, contribution_weighted_uncertainty

In [ ]:
def compute_auroc(labels: np.ndarray, scores: np.ndarray, score_name: str, task_name: str):
    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())

    if n_pos == 0 or n_neg == 0:
        print(f"[{score_name}] Cannot compute {task_name} AUROC: n_positive={n_pos}, n_negative={n_neg}")
        return None

    auroc = roc_auc_score(labels, scores)
    print(f"[{score_name}] {task_name} AUROC: {auroc:.4f} (n_negative={n_neg}, n_positive={n_pos})")
    return auroc

In [ ]:
def evaluate_model(model, dataloader, device, num_classes=8, unk_label=8):

    model.eval()

    all_targets = []
    all_preds = []
    all_contrib_uncertainty = []

    with torch.no_grad():
        for inputs, targets in tqdm(
            dataloader,
            desc="Evaluating",
            ncols=100,
        ):
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Single forward pass for the complete batch
            alpha = model.get_output(inputs)

            predicted, _, _, contrib_uncertainty = (
                get_predictions_and_uncertainty(model, alpha)
            )

            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_contrib_uncertainty.extend(
                contrib_uncertainty.cpu().numpy()
            )

    all_targets = np.asarray(all_targets)
    all_preds = np.asarray(all_preds)
    all_contrib_uncertainty = np.asarray(all_contrib_uncertainty)


    # 1. Classification metrics
    known_mask = all_targets != unk_label

    known_targets = all_targets[known_mask]
    known_preds = all_preds[known_mask]

    macro_f1 = f1_score(
        known_targets,
        known_preds,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc = balanced_accuracy_score(
        known_targets,
        known_preds,
    )


    # 2. UNK-detection AUROC
    is_unk = (all_targets == unk_label).astype(int)

    unk_auroc = compute_auroc(
        is_unk,
        all_contrib_uncertainty,
        "contribution-weighted avg",
        "UNK-detection",
    )


    return macro_f1, balanced_acc, unk_auroc


In [ ]:
# Test evaluation
(
    test_macro_f1,
    test_balanced_acc,
    test_unk_auroc,
) = evaluate_model(
    model,
    test_loader,
    device,
)

print(f"Macro F1:                         {test_macro_f1:.4f}")
print(f"Balanced Accuracy:                {test_balanced_acc:.4f}")
print(f"UNK-detection AUROC:              {test_unk_auroc:.4f}")

In [ ]:
@torch.no_grad()
def collect_test_predictions(model, dataloader, device):

    model.eval()

    all_preds = []
    all_targets = []
    all_uncertainty = []

    for inputs, targets in tqdm(
        dataloader,
        desc="Collecting test predictions",
        ncols=100,
    ):
        inputs = inputs.to(device)
        targets = targets.to(device)

        alpha = model.get_output(inputs)

        preds, _, _, contrib_uncertainty = (
            get_predictions_and_uncertainty(model, alpha)
        )

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_uncertainty.extend(
            contrib_uncertainty.cpu().numpy()
        )

    return (
        np.asarray(all_preds),
        np.asarray(all_targets),
        np.asarray(all_uncertainty),
    )


def evaluate_id_after_pruning(
    preds,
    targets,
    uncertainty,
    prune_frac=0.10,
    num_classes=8,
    unk_label=8,
):
    """
    Reject the most uncertain `prune_frac` of ID samples and recompute
    Macro F1 and Balanced Accuracy on the retained ID samples.
    """

    id_mask = targets != unk_label

    preds_id = preds[id_mask]
    targets_id = targets[id_mask]
    uncertainty_id = uncertainty[id_mask]

    n_id = len(targets_id)
    n_prune = int(round(n_id * prune_frac))
    n_keep = n_id - n_prune

    # Ascending uncertainty:
    # least uncertain samples are retained.
    order = np.argsort(uncertainty_id)
    keep_idx = order[:n_keep]

    kept_preds = preds_id[keep_idx]
    kept_targets = targets_id[keep_idx]

    macro_f1_before = f1_score(
        targets_id,
        preds_id,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc_before = balanced_accuracy_score(
        targets_id,
        preds_id,
    )

    macro_f1_after = f1_score(
        kept_targets,
        kept_preds,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc_after = balanced_accuracy_score(
        kept_targets,
        kept_preds,
    )

    print(f"[ID-only: reject {prune_frac:.0%} most uncertain]")
    print(
        f"  N ID: {n_id} | "
        f"N retained: {n_keep} | "
        f"N rejected: {n_prune}"
    )
    print(
        f"  Macro F1:          "
        f"{macro_f1_before:.4f} -> {macro_f1_after:.4f}"
    )
    print(
        f"  Balanced Accuracy: "
        f"{balanced_acc_before:.4f} -> {balanced_acc_after:.4f}"
    )

    return {
        "n_id": n_id,
        "n_retained": n_keep,
        "n_rejected": n_prune,
        "macro_f1_before": macro_f1_before,
        "macro_f1_after": macro_f1_after,
        "balanced_acc_before": balanced_acc_before,
        "balanced_acc_after": balanced_acc_after,
    }



def evaluate_full_pruning_composition(
    targets,
    uncertainty,
    prune_frac=0.10,
    unk_label=8,
):
    """
    Reject the most uncertain `prune_frac` of the complete test set
    (ID + UNK) and report the fraction of UNK samples rejected.
    """

    n_total = len(targets)
    n_prune = int(round(n_total * prune_frac))

    # Reject the most uncertain samples
    rejected_idx = np.argsort(-uncertainty)[:n_prune]

    is_unk = targets == unk_label
    n_unk = int(is_unk.sum())
    n_rejected_unk = int(is_unk[rejected_idx].sum())

    unk_caught = (
        n_rejected_unk / n_unk
        if n_unk > 0
        else np.nan
    )

    print(f"[Full test set: reject {prune_frac:.0%} most uncertain]")
    print(f"  N total: {n_total} | N rejected: {n_prune}")
    print(
        f"  UNK caught: {n_rejected_unk}/{n_unk} "
        f"({unk_caught:.2%})"
    )

    return {
        "n_total": n_total,
        "n_rejected": n_prune,
        "unk_caught": unk_caught,
        "n_rejected_unk": n_rejected_unk,
    }


In [ ]:
# Collect predictions 

preds_full, targets_full, uncertainty_full = collect_test_predictions(
    model,
    test_loader,
    device,
)


# Reject 10% most uncertain ID samples

id_results = evaluate_id_after_pruning(
    preds_full,
    targets_full,
    uncertainty_full,
    prune_frac=0.10,
    num_classes=8,
    unk_label=8,
)

# Reject 10% most uncertain samples from complete test set

full_results = evaluate_full_pruning_composition(
    targets_full,
    uncertainty_full,
    prune_frac=0.10,
    unk_label=8,
)